# PathSnap 2 · Bayesian tuning and Viterbi decoding

This notebook reruns the ten Bayesian trials and route reconstruction. It uses the saved untuned classifier as the baseline and the original five trip folds. It can run independently of notebook 1.

Use Python 3.13. The setup cell installs the training dependencies into the selected notebook kernel if any are missing. If you already ran a cell that imported NumPy, restart the kernel before running all cells.

In [ ]:
from importlib.util import find_spec
import subprocess
import sys

packages = {
    "numpy": "numpy==2.1.3",
    "scipy": "scipy==1.15.3",
    "sklearn": "scikit-learn==1.6.1",
    "joblib": "joblib==1.4.2",
    "threadpoolctl": "threadpoolctl==3.5.0",
}
if any(find_spec(name) is None for name in packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages.values()])
else:
    print("Training dependencies are installed in this kernel.")

In [ ]:
from pathlib import Path
import csv
import hashlib
import os
import subprocess
import sys
import tempfile
from IPython.display import Markdown, display

# Set these paths if the notebook or dataset is outside the extracted project.
PROJECT_ROOT = None
DATASET_ZIP = None
cwd = Path.cwd()
if PROJECT_ROOT is None:
    PROJECT_ROOT = next((p for p in (cwd, *cwd.parents)
                         if (p / "stage-4" / "compare.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT above to the extracted PathSnap folder.")
root = Path(PROJECT_ROOT).expanduser().resolve()
archive = (Path(DATASET_ZIP).expanduser().resolve() if DATASET_ZIP is not None
           else root / "inputs" / "Student folder.zip")
if not archive.is_file():
    raise FileNotFoundError(f"Place the original student dataset ZIP here: {archive}")

work = root / "work" / "notebook-runs"
work.mkdir(parents=True, exist_ok=True)
print("Dataset:", archive)

def run_python(*args):
    command = [sys.executable, "-u", *map(str, args)]
    env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", LOKY_MAX_CPU_COUNT="3")
    with subprocess.Popen(command, cwd=root, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace") as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        exit_code = process.wait()
    if exit_code:
        raise subprocess.CalledProcessError(exit_code, command)

def read_rows(file):
    with file.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))

def show_table(headers, rows):
    lines = [headers, ["---"] * len(headers), *rows]
    display(Markdown("\n".join("| " + " | ".join(map(str, row)) + " |" for row in lines)))

## Build or reuse the candidate features

This uses the original preprocessing and scoring rules. Reference-derived labels are stored separately from GPS and road features.

In [ ]:
import numpy
import scipy
import sklearn
import joblib

print(f"NumPy {numpy.__version__}, SciPy {scipy.__version__}, "
      f"scikit-learn {sklearn.__version__}, joblib {joblib.__version__}")
stage3 = root / "stage-3"
data_hash = hashlib.sha256(archive.read_bytes()).hexdigest()[:12]
source_hash = hashlib.sha256(b"".join(
    (stage3 / name).read_bytes()
    for name in ("candidates.py", "geometry.py", "reference.py", "run_stage3.py")
)).hexdigest()[:12]
cache = work / f"cache-{data_hash}-{source_hash}"
marker = cache / "complete"
if not marker.is_file():
    cache.mkdir(parents=True, exist_ok=True)
    run_python(stage3 / "run_stage3.py", archive, "--out", cache / "baseline", "--cache", cache)
    marker.touch()
else:
    print("Reusing candidate features from this dataset and source revision.")
print("Candidate cache:", cache)

## Tune histogram gradient boosting

The objective is mean trip-local Macro F1 on the training folds. The winning configuration is refitted on all 80 training trips.

In [ ]:
import json
from shutil import copy2, copytree

source = root / "stage-5"
tuning_dir = Path(tempfile.mkdtemp(prefix="tuning-and-decoding-", dir=work))
for name in ("tune.py", "evaluate.py", "verify.py", "inference.py",
             "decoder.py", "untuned.joblib"):
    copy2(source / name, tuning_dir / name)
copytree(source / "shared", tuning_dir / "shared")
scratch = tuning_dir / "scratch"

run_python(tuning_dir / "tune.py", archive, cache, scratch)
settings = json.loads((tuning_dir / "tuning.json").read_text())
print("Winning settings:", settings["selected"])
print(f"Training-fold Macro F1: {settings['selected_cv_macro_f1']:.5f}")

## Evaluate Viterbi and verify predictions

The decoder scale is selected on held-out training predictions before scoring the student-test trips. Verification replays all ten trips from raw GPS and road data.

In [ ]:
run_python(tuning_dir / "evaluate.py", archive, cache, scratch)
run_python(tuning_dir / "verify.py", archive, cache)

## Accuracies from this run

Point accuracy covers eligible GPS points. Sequence F1 and continuity cover the full predicted route.

In [ ]:
methods = {"untuned": "Untuned classifier", "tuned": "Tuned classifier",
           "viterbi": "Tuned classifier + Viterbi"}
rows = [
    [methods[r["method"]], f'{float(r["accuracy"]):.2%}',
     f'{int(r["correct"]):,} / {int(r["eligible"]):,}', f'{float(r["macro_f1"]):.4f}',
     f'{float(r["sequence_f1"]):.4f}', f'{float(r["continuity"]):.2%}']
    for r in read_rows(tuning_dir / "comparison.csv") if r["split"] == "student_test"
]
show_table(["Pipeline", "Point accuracy", "Correct / eligible",
            "Macro F1", "Sequence F1", "Continuity"], rows)
print("Fresh results and tuned model:", tuning_dir)